In [ ]:
pip install pyswarms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.5 MB/s eta 0:00:00


In [2]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0, MobileNet
from transformers import ViTFeatureExtractor, TFViTModel

from tensorflow.keras.preprocessing.image import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pyswarms as ps

from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from PIL import Image

In [3]:
INPUT_SHAPE = 224

In [4]:
import cv2
import numpy as np
import random
from skimage.util import random_noise
from skimage import exposure

# --- Augmentation Functions ---

def add_gaussian_noise(image):
    """Adds Gaussian noise to an image."""
    # Ensure image is float type for noise addition
    img_float = image.astype(np.float64) / 255.0
    # Adjust variance (0.1**0.5 here) as needed
    noise = np.random.normal(0, 0.1**0.5, img_float.shape)
    noisy_image = np.clip(img_float + noise, 0, 1)
    return (noisy_image * 255).astype(np.uint8)

def change_brightness_contrast(image, alpha=None, beta=None):
    """
    Adjusts brightness and contrast.
    alpha: Contrast control (e.g., 0.8-1.2). If None, random value is chosen.
    beta: Brightness control (e.g., -20 to 20). If None, random value is chosen.
    """
    if alpha is None:
        alpha = random.uniform(0.8, 1.2) # Slight contrast variation
    if beta is None:
        beta = random.randint(-20, 20)   # Slight brightness variation

    # Apply the formula: new_image = alpha * original_image + beta
    new_image = np.clip(alpha * image.astype(np.float32) + beta, 0, 255).astype(np.uint8)
    return new_image

def affine_transform(image, angle=None, tx=None, ty=None, shear=None, zoom=None):
    """
    Applies random affine transformations: rotation, translation.
    Uses cv2.BORDER_REFLECT for handling borders.
    Note: Shear and zoom are commented out for simplicity but can be added.
    """
    rows, cols, _ = image.shape
    center = (cols / 2, rows / 2)

    # Default border mode
    border_mode = cv2.BORDER_REFLECT # Use cv2.BORDER_REFLECT instead of BORDER_REFLECT_1

    # Rotation
    if angle is None:
        angle = random.uniform(-15, 15) # Random angle between -15 and 15 degrees
    M_rot = cv2.getRotationMatrix2D(center, angle, 1)

    # Translation
    if tx is None:
        tx = random.uniform(-0.1, 0.1) * cols # Translate by max 10% of width
    if ty is None:
        ty = random.uniform(-0.1, 0.1) * rows # Translate by max 10% of height
    # Translation requires a 2x3 matrix
    M_trans = np.float32([[1, 0, tx], [0, 1, ty]])

    # --- Apply transformations sequentially ---
    # Apply Rotation first
    # Use the chosen border mode
    rotated_img = cv2.warpAffine(image, M_rot, (cols, rows), flags=cv2.INTER_LINEAR, borderMode=border_mode)

    # Apply Translation to the rotated image
    # Use the chosen border mode
    translated_img = cv2.warpAffine(rotated_img, M_trans, (cols, rows), flags=cv2.INTER_LINEAR, borderMode=border_mode)

    # --- Optional: Add Shear and Zoom ---
    # Shear requires careful matrix construction or separate warping
    # Zoom can use getRotationMatrix2D with angle=0 and a zoom factor

    return translated_img # Return the result of combined transforms

def horizontal_flip(image):
    """Applies horizontal flip."""
    return cv2.flip(image, 1)

def vertical_flip(image):
    """Applies vertical flip."""
    return cv2.flip(image, 0)

In [5]:
import os
import glob
import cv2
import numpy as np
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator # For Stage 2 explanation

# Assume augmentation helper functions (add_gaussian_noise, change_brightness_contrast,
# affine_transform, horizontal_flip, vertical_flip) from the previous block are defined here or imported.

def load_and_augment_dataset(dataset_path, augment_factor=2):
    """
    Traverses the dataset directory, loads .bmp images, and applies offline
    augmentations to increase dataset size.

    Args:
        dataset_path (str): Path to the root dataset directory.
        augment_factor (int): The factor by which to increase the dataset size
                              (e.g., 6 means 1 original + 5 augmented images per original).

    Returns:
        tuple: (list_of_augmented_images, list_of_corresponding_labels)
    """
    classes = ["im_Dyskeratotic", "im_Koilocytotic", "im_Metaplastic", "im_Parabasal", "im_Superficial-Intermediate"]
    original_images = []
    original_labels = []
    print("--- Loading Original Images ---")
    for cls in classes:
        # Adjust the path structure based on your actual dataset layout
        # Example assumes: dataset_path/im_Dyskeratotic/im_Dyskeratotic/CROPPED/*.bmp
        folder_path = os.path.join(dataset_path, cls, cls, "CROPPED")
        file_list = glob.glob(os.path.join(folder_path, "*.bmp"))
        print(f"Found {len(file_list)} original images in {folder_path}")
        for file in file_list:
            img = cv2.imread(file)
            if img is None:
                print(f"Warning: Could not read image {file}. Skipping.")
                continue
            # Convert BGR (OpenCV default) to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            original_images.append(img)
            original_labels.append(cls) # Use the folder name as the label

    print(f"\n--- Applying Offline Augmentations (Target Factor: {augment_factor}x) ---")
    augmented_images = []
    augmented_labels = []

    # Define the pool of augmentation functions to choose from
    # Excluding Canny/Laplace based on previous notes, but you can add them back if desired.
    available_augmentations = [
        add_gaussian_noise,
        change_brightness_contrast,
        lambda i: affine_transform(i, angle=random.uniform(-15, 15)), # Rotation only
        lambda i: affine_transform(i, tx=random.uniform(-0.1, 0.1)*i.shape[1], ty=random.uniform(-0.1, 0.1)*i.shape[0]), # Translation only
        # lambda i: affine_transform(i, shear=random.uniform(-0.1, 0.1)), # Shear - uncomment if needed
        # lambda i: affine_transform(i, zoom=random.uniform(0.9, 1.1)), # Zoom - uncomment if needed
        horizontal_flip,
        # vertical_flip # Vertical flip might not be suitable for cell images
    ]

    total_original = len(original_images)
    for i, (img, label) in enumerate(zip(original_images, original_labels)):
        if (i + 1) % 100 == 0:
            print(f"Processing original image {i+1}/{total_original}...")

        # 1. Add the original image
        augmented_images.append(img)
        augmented_labels.append(label)

        # 2. Add augmented versions
        augmentations_to_add = augment_factor - 1
        applied_count = 0
        attempts = 0 # To prevent infinite loops if augmentations fail
        max_attempts = augmentations_to_add * 3 # Allow some retries

        # Use a copy of the available functions list for this image to avoid duplicates if possible
        current_augment_pool = available_augmentations[:]

        while applied_count < augmentations_to_add and attempts < max_attempts:
            attempts += 1
            if not current_augment_pool: # Reset pool if we've used all unique ones
                 current_augment_pool = available_augmentations[:]

            # Select and remove a random augmentation function from the pool for this image
            aug_func_index = random.randrange(len(current_augment_pool))
            aug_func = current_augment_pool.pop(aug_func_index)

            try:
                # Apply augmentation to a copy of the original image
                augmented_img = aug_func(img.copy())

                # Basic check for validity (e.g., ensure it's not None and has the right dimensions)
                if augmented_img is not None and augmented_img.shape == img.shape and augmented_img.dtype == img.dtype:
                    augmented_images.append(augmented_img)
                    augmented_labels.append(label)
                    applied_count += 1
                else:
                    print(f"Warning: Augmentation {getattr(aug_func, '__name__', 'lambda')} produced invalid output for image {i}. Retrying.")

            except Exception as e:
                print(f"Warning: Augmentation {getattr(aug_func, '__name__', 'lambda')} failed for image {i}: {e}. Retrying.")

        if applied_count < augmentations_to_add:
             print(f"Warning: Only added {applied_count}/{augmentations_to_add} augmentations for image {i} after {attempts} attempts.")


    print(f"\n--- Augmentation Summary ---")
    print(f"Original images: {len(original_images)}")
    print(f"Total images after augmentation: {len(augmented_images)}")
    print(f"Labels generated: {len(augmented_labels)}")

    # Shuffle the augmented dataset
    combined = list(zip(augmented_images, augmented_labels))
    random.shuffle(combined)
    augmented_images[:], augmented_labels[:] = zip(*combined)

    return augmented_images, augmented_labels

# --- How to Use ---
# dataset_main_path = '/kaggle/input/cervical-cancer-largest-dataset-sipakmed'
# augmented_images, augmented_labels = load_and_augment_dataset(dataset_main_path, augment_factor=6)

# print(f"Total augmented images loaded: {len(augmented_images)}")

# --- Explanation of Stage 2 (ImageDataGenerator) ---
# Keras ImageDataGenerator is typically used for *online* data augmentation
# during the training of a deep learning model (like EfficientNet, MobileNet, ViT itself).
# Example:
# datagen = ImageDataGenerator(
#     rotation_range=20,
#     width_shift_range=0.1,
#     height_shift_range=0.1,
#     shear_range=0.1,
#     zoom_range=0.1,
#     horizontal_flip=True,
#     fill_mode='nearest'
# )

# If you were training a Keras model directly:
# model.fit(datagen.flow(X_train_images, y_train_labels, batch_size=32), ...)

# In your current workflow:
# 1. You load images.
# 2. You apply *offline* augmentations (using the functions above).
# 3. You resize images (preprocess_images).
# 4. You extract features using pre-trained models (extract_features).
# 5. You potentially apply PCA/PSO to these *features*.
# 6. You train a classifier (SVM, RF) on the *final features*.

# Because steps 5 and 6 work on static features, ImageDataGenerator isn't directly
# applied *during* the SVM/RF training. However, the extensive offline augmentation
# in step 2 ensures that the features extracted in step 4 are learned from a much
# more diverse set of input images, achieving the goal of making the model more robust.

In [6]:
def load_dataset(dataset_path):
    """
    Traverses the dataset directory structure and loads only .bmp images.
    Expected structure:
      dataset_path/
         Dyskeratotic/Dyskeratotic/CROPPED/*.bmp
         Koilocytotic/Koilocytotic/CROPPED/*.bmp
         Metaplastic/Metaplastic/CROPPED/*.bmp
         Parabasal/Parabasal/CROPPED/*.bmp
         Superficial-Intermediate/Superficial-Intermediate/CROPPED/*.bmp
    """
    classes = ["im_Dyskeratotic", "im_Koilocytotic", "im_Metaplastic", "im_Parabasal", "im_Superficial-Intermediate"]
    images = []
    labels = []
    for cls in classes:
        folder_path = os.path.join(dataset_path, cls, cls, "CROPPED")
        # Only select .bmp files
        file_list = glob.glob(os.path.join(folder_path, "*.bmp"))
        print(f"Found {len(file_list)} images in {folder_path}")
        for file in file_list:
            img = cv2.imread(file)
            if img is None:
                continue
            # Convert BGR to RGB for consistency with most DL models
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(cls)
    return images, labels

In [7]:
# 2. Preprocessing (resize & convert to array)
def preprocess_images(images, target_size=(INPUT_SHAPE, INPUT_SHAPE)):
    proc_imgs = []
    for img in images:
        # Resize image to model input size
        img_resized = cv2.resize(img, target_size)
        img_array = img_to_array(img_resized)
        proc_imgs.append(img_array)
    return np.array(proc_imgs)

In [8]:
def extract_features(images, model_name='efficientnet'):
    if model_name.lower() == 'efficientnet':
        # Use EfficientNetB0 with global average pooling
        base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg', input_shape=(INPUT_SHAPE,INPUT_SHAPE,3))
        preproc_fn = tf.keras.applications.efficientnet.preprocess_input
    elif model_name.lower() == 'mobilenet':
        base_model = MobileNet(weights='imagenet', include_top=False, pooling='avg', input_shape=(INPUT_SHAPE,INPUT_SHAPE,3))
        preproc_fn = tf.keras.applications.mobilenet.preprocess_input
    else:
        raise ValueError("Invalid model name. Choose 'efficientnet' or 'mobilenet'.")
    
    # Preprocess images as required by the model
    images_preprocessed = preproc_fn(images)
    
    # Extract features
    features = base_model.predict(images_preprocessed, verbose=1)
    return features

In [9]:
# 4. PSO Objective Function for Feature Selection
def objective_function(mask, X, y):
    """
    For each particle, select features based on the binary mask and evaluate the performance
    of an SVM classifier. The objective is to minimize (1 - accuracy).
    """
    n_particles = mask.shape[0]
    scores = np.zeros(n_particles)
    for i in range(n_particles):
        binary_mask = mask[i].astype(bool)
        # If no feature is selected, assign worst cost (1.0)
        if np.sum(binary_mask) == 0:
            scores[i] = 1.0
        else:
            # Select features based on mask
            X_selected = X[:, binary_mask]
            # Use a fixed random state for reproducibility
            X_train, X_val, y_train, y_val = train_test_split(X_selected, y, test_size=0.3, random_state=42)
            clf = SVC(kernel='linear', random_state=42)
            clf.fit(X_train, y_train)
            acc = clf.score(X_val, y_val)
            scores[i] = 1 - acc  # lower is better
    return scores

# def objective_function(mask, X, y):
#     n_particles = mask.shape[0]
#     scores = np.zeros(n_particles)
#     for i in range(n_particles):
#         binary_mask = mask[i].astype(bool)
#         if np.sum(binary_mask) == 0:
#             scores[i] = 1.0
#         else:
#             X_selected = X[:, binary_mask]
#             X_train, X_val, y_train, y_val = train_test_split(X_selected, y, test_size=0.3, random_state=42)

#             # Convert to cuML DataFrames (necessary for cuML)
#             X_train_cuml = cuml.DataFrame.from_pandas(pd.DataFrame(X_train))
#             X_val_cuml = cuml.DataFrame.from_pandas(pd.DataFrame(X_val))
#             y_train_cuml = cuml.Series(y_train)
#             y_val_cuml = cuml.Series(y_val)

#             clf = SVC(kernel='linear', random_state=42)
#             clf.fit(X_train_cuml, y_train_cuml)
#             acc = clf.score(X_val_cuml, y_val_cuml)
#             scores[i] = 1 - acc
#     return scores

In [10]:
# images,labels = load_dataset('/kaggle/input/cervical-cancer-largest-dataset-sipakmed')
images,labels = load_and_augment_dataset('/kaggle/input/cervical-cancer-largest-dataset-sipakmed',3)

--- Loading Original Images ---
Found 813 original images in /kaggle/input/cervical-cancer-largest-dataset-sipakmed/im_Dyskeratotic/im_Dyskeratotic/CROPPED
Found 825 original images in /kaggle/input/cervical-cancer-largest-dataset-sipakmed/im_Koilocytotic/im_Koilocytotic/CROPPED
Found 793 original images in /kaggle/input/cervical-cancer-largest-dataset-sipakmed/im_Metaplastic/im_Metaplastic/CROPPED
Found 787 original images in /kaggle/input/cervical-cancer-largest-dataset-sipakmed/im_Parabasal/im_Parabasal/CROPPED
Found 831 original images in /kaggle/input/cervical-cancer-largest-dataset-sipakmed/im_Superficial-Intermediate/im_Superficial-Intermediate/CROPPED

--- Applying Offline Augmentations (Target Factor: 3x) ---
Processing original image 100/4049...
Processing original image 200/4049...
Processing original image 300/4049...
Processing original image 400/4049...
Processing original image 500/4049...
Processing original image 600/4049...
Processing original image 700/4049...
Proces

In [11]:
# Convert labels to a numpy array
y = np.array(labels)
np.save('labels.npy',y)

In [12]:
images_proc = preprocess_images(images, target_size=(224, 224))

In [13]:
# Batch Feature Extraction Function Using ViT
def extract_features_batch(images, batch_size=16):
    """
    Extract features from images using a Vision Transformer (ViT) in batches.
    """
    # Load ViT feature extractor and model
    feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
    vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
    
    features_list = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        # Convert each image to PIL format
        pil_images = [Image.fromarray(img.astype('uint8'), 'RGB') for img in batch]
        inputs = feature_extractor(images=pil_images, return_tensors="tf")
        outputs = vit_model(inputs['pixel_values'])
        # Extract the [CLS] token embedding for each image
        features_batch = outputs.last_hidden_state[:, 0, :].numpy()
        features_list.append(features_batch)
    features = np.vstack(features_list)
    return features

In [14]:
def extract_features(images, model_name='vit'):
    if model_name.lower() == 'efficientnet':
        preproc_fn = tf.keras.applications.efficientnet.preprocess_input
        base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='max', input_shape=(224,224,3))
        images_preprocessed = preproc_fn(images)
        features = base_model.predict(images_preprocessed, verbose=1)
        
    elif model_name.lower() == 'mobilenet':
        preproc_fn = tf.keras.applications.mobilenet.preprocess_input
        base_model = MobileNet(weights='imagenet', include_top=False, pooling='max', input_shape=(224,224,3))
        images_preprocessed = preproc_fn(images)
        features = base_model.predict(images_preprocessed, verbose=1)
        
    elif model_name.lower() == 'vit':
        features = extract_features_batch(images_proc, batch_size=64)
    else:
        raise ValueError("Invalid model name. Choose 'efficientnet', 'mobilenet', or 'vit'.")
    
    return features

In [15]:
# Load features from all models
features_efficientnet = extract_features(images_proc, model_name='efficientnet')
np.save('features_efficientnet.npy',features_efficientnet)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
380/380 ━━━━━━━━━━━━━━━━━━━━ 29s 50ms/step


In [16]:
features_mobilenet = extract_features(images_proc, model_name='mobilenet')
np.save('features_mobilenet.npy',features_mobilenet)

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
380/380 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step


In [17]:
features_vit = extract_features(images_proc, model_name='vit')
np.save('features_vit.npy',features_vit)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFViTModel.

All the weights of TFViTModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTModel for predictions without further training.


In [18]:
# Concatenate features from all models
combined_features = np.concatenate([features_efficientnet, features_mobilenet, features_vit], axis=1)

In [19]:
np.save('combined_features.npy',combined_features)

In [20]:
# Apply PCA for dimensionality reduction
pca = PCA(0.99)
pca.fit(combined_features)
pca_features = pca.fit_transform(combined_features)

In [21]:
# Convert labels to a numpy array
y = np.array(labels)
np.save('labels.npy',y)
X = pca_features  # feature matrix

In [22]:
# Feature Selection via PSO

# PSO hyperparameters
options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 3, 'p': 2}
dimensions = X.shape[1]  # total number of features extracted
print(dimensions)

# Use BinaryPSO (for binary selection: 1=select feature, 0=discard)
optimizer = ps.discrete.BinaryPSO(n_particles=20, dimensions=dimensions, options=options)

1632


In [23]:
# # Run PSO: note that the objective_function receives X and y via additional arguments
# best_cost, best_pos = optimizer.optimize(objective_function, iters=50, X=X, y=y, verbose=True)
# print("Best PSO cost (1 - best accuracy):", best_cost)
# print("Number of selected features:", np.sum(best_pos))

In [24]:
# # use the selected features
# selected_features = best_pos.astype(bool)
# X_selected = X[:, selected_features]

In [25]:
 # Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(pca_features, y, test_size=0.2, random_state=42)

In [26]:
# Train SVM classifier
clf = SVC(kernel='rbf', random_state=42)
clf.fit(X_train, y_train)

SVC(random_state=42)

In [27]:
# Evaluate classifier
y_pred = clf.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
                             precision    recall  f1-score   support

            im_Dyskeratotic       0.95      0.99      0.97       502
            im_Koilocytotic       0.97      0.92      0.94       517
             im_Metaplastic       0.95      0.96      0.96       464
               im_Parabasal       0.97      0.97      0.97       478
im_Superficial-Intermediate       0.99      0.99      0.99       469

                   accuracy                           0.97      2430
                  macro avg       0.97      0.97      0.97      2430
               weighted avg       0.97      0.97      0.96      2430



In [28]:
# def svm_objective(params):
#     C, gamma = params
#     clf = SVC(C=C, gamma=gamma)
#     score = cross_val_score(clf, pca_features, labels, cv=5).mean()
#     return -score

# lb, ub = [0.01, 1e-5], [100, 1]
# best_params, _ = pso(svm_objective, lb, ub)
# print("Best SVM Parameters:", best_params)

In [29]:
X = np.load('/kaggle/input/mobilenet-artifacts/features.npy')
y=labels

In [30]:
# best_pos = np.load('/kaggle/input/mobilenet-artifacts/pso_mobilenet.npy')
# selected_features = best_pos.astype(bool)

In [31]:
# X=X[:,selected_features]

In [32]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb

In [34]:
# 1. Import necessary libraries
import xgboost as xgb
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder # <-- Import LabelEncoder
# Assuming X_train, y_train, X_test, y_test are already defined and loaded
# from sklearn.model_selection import train_test_split
# from sklearn.datasets import make_classification

# Example: Create dummy data if you don't have it loaded (replace with your actual data)
# X, y_strings = make_classification(n_samples=1000, n_features=20, n_informative=15,
#                                     n_redundant=5, n_classes=5, n_clusters_per_class=1, random_state=42)
# label_map = {0: 'im_Dyskeratotic', 1: 'im_Koilocytotic', 2: 'im_Metaplastic', 3: 'im_Parabasal', 4: 'im_Superficial-Intermediate'}
# y = [label_map[label] for label in y_strings] # Create string labels
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


# 2. Encode the string labels into integers
le = LabelEncoder()
# Fit on the training labels AND transform them
y_train_encoded = le.fit_transform(y_train)
# ONLY transform the test labels using the same fitted encoder
y_test_encoded = le.transform(y_test)

# You can check the mapping:
# print("Original classes:", le.classes_)
# print("Encoded classes:", le.transform(le.classes_))
# print("First 5 original y_train:", y_train[:5])
# print("First 5 encoded y_train:", y_train_encoded[:5])


# 3. Instantiate the XGBoost Classifier
clf = xgb.XGBClassifier(
    random_state=42,
    use_label_encoder=False, # Keep this as False
    # Use 'mlogloss' for multi-class classification as the eval_metric
    eval_metric='mlogloss'
    # Add other hyperparameters as needed (n_estimators, learning_rate, etc.)
)

# 4. Train the classifier using the ENCODED training labels
print("Starting XGBoost training...")
clf.fit(X_train, y_train_encoded) # <-- Use y_train_encoded
print("Training finished.")

# 5. Make predictions on the test set
# The model will predict the ENCODED labels
y_pred_encoded = clf.predict(X_test)

# 6. Evaluate the predictions
print("\n--- XGBoost Classification Report ---")
# Option 1: Report using the original string labels (more readable)
# Decode the predictions back to original string labels for the report
y_pred_decoded = le.inverse_transform(y_pred_encoded)
# Compare original test labels (y_test) with decoded predictions (y_pred_decoded)
# Ensure target_names are correctly specified using the encoder's learned classes
print(classification_report(y_test, y_pred_decoded, target_names=le.classes_))

# Option 2: Report using the encoded integer labels
# print("\n--- XGBoost Classification Report (Encoded Labels) ---")
# print(classification_report(y_test_encoded, y_pred_encoded))

Starting XGBoost training...
Training finished.

--- XGBoost Classification Report ---
                             precision    recall  f1-score   support

            im_Dyskeratotic       0.90      0.97      0.94       502
            im_Koilocytotic       0.92      0.88      0.90       517
             im_Metaplastic       0.91      0.93      0.92       464
               im_Parabasal       0.96      0.92      0.94       478
im_Superficial-Intermediate       0.98      0.98      0.98       469

                   accuracy                           0.93      2430
                  macro avg       0.94      0.93      0.93      2430
               weighted avg       0.93      0.93      0.93      2430



In [35]:
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

                             precision    recall  f1-score   support

            im_Dyskeratotic       0.80      0.77      0.79       502
            im_Koilocytotic       0.64      0.63      0.64       517
             im_Metaplastic       0.66      0.68      0.67       464
               im_Parabasal       0.78      0.81      0.79       478
im_Superficial-Intermediate       0.89      0.88      0.89       469

                   accuracy                           0.76      2430
                  macro avg       0.76      0.76      0.76      2430
               weighted avg       0.76      0.76      0.76      2430



In [36]:
clf = RandomForestClassifier(n_estimators = 200,bootstrap=True)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

                             precision    recall  f1-score   support

            im_Dyskeratotic       0.85      0.89      0.87       502
            im_Koilocytotic       0.86      0.74      0.80       517
             im_Metaplastic       0.81      0.83      0.82       464
               im_Parabasal       0.88      0.86      0.87       478
im_Superficial-Intermediate       0.86      0.95      0.90       469

                   accuracy                           0.85      2430
                  macro avg       0.85      0.85      0.85      2430
               weighted avg       0.85      0.85      0.85      2430

